# Notebook 02 — Kafka anatomy: topics, partitions, offsets

**Goal:** by the end you can draw Kafka's internal structure from memory, and you will have *seen* every part of it: in commands, and as real files on disk.

**Needs:** Notebook 01 finished (Kafka container `kafka-lab` is running).

| Step | What you do |
|---|---|
| 0 | Connect this notebook to Kafka |
| 1 | Learn the 5 words: message, topic, partition, offset, segment |
| 2 | Create a topic |
| 3 | Inspect it |
| 4 | Put 6 messages into it |
| 5 | See offsets |
| 6 | Read a message by its address |
| 7 | Open the files on disk |
| 8 | Decode a file to see messages inside |
| 9 | Watch a log split into segments |
| 10 | Retention: how Kafka deletes |

## Step 0 — Connect

Run this cell first. It defines three small helpers used everywhere below:
- `run_on_host(command)` — run a command on your real machine (needed because VS Code is sandboxed; explained in Notebook 01).
- `kafka_tool(tool, *args)` — run one of Kafka's own command-line tools **inside** the `kafka-lab` container.
- `admin` — a Python object for asking Kafka questions and creating topics.

In [2]:
import subprocess
import time
from pathlib import Path

from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = "localhost:9092"          # where Python knocks (see Notebook 01, Step 4)
CONTAINER = "kafka-lab"
DATA_DIR = "/var/lib/kafka/data"      # where Kafka stores messages, inside the container


def run_on_host(command, show=True):
    """Run a command on your real machine; print its output if show=True; return the process."""
    in_flatpak_sandbox = Path("/.flatpak-info").exists()
    full_command = ["flatpak-spawn", "--host", *command] if in_flatpak_sandbox else command
    result = subprocess.run(full_command, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if output and show:
        print(output)
    return result


def kafka_tool(tool, *args, show=True):
    """Run a Kafka command-line tool inside the container (Kafka ships these tools)."""
    return run_on_host(["docker", "exec", CONTAINER, f"/opt/kafka/bin/{tool}", *args], show=show)


def in_container(*command):
    """Run an ordinary Linux command (ls, etc.) inside the container."""
    return run_on_host(["docker", "exec", CONTAINER, *command])


admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
cluster = admin.list_topics(timeout=10)
print("Connected. Brokers:", [f"id={b.id}" for b in cluster.brokers.values()])

Connected. Brokers: ['id=1']


In [5]:
cluster.brokers.values()

dict_values([BrokerMetadata(1, localhost:9092)])

In [6]:
admin.list_topics(timeout=10)

ClusterMetadata(5L6g3nShT-eMCtK--X86sw)

## Step 1 — Five words

```text
  CLUSTER (all your Kafka servers)
  └── BROKER 1 (one server)
      └── TOPIC "orders"            <- a named stream, like a folder
          ├── PARTITION 0   [ A ][ B ][ D ]            <- an ordered list, only ever appended to
          │                  0    1    2               <- OFFSETS (position numbers)
          ├── PARTITION 1   [ C ][ F ]
          │                  0    1
          └── PARTITION 2   [ E ]
                             0
```

| Word | Meaning | Remember this |
|---|---|---|
| **Message** (also *record* or *event*) | One piece of data, e.g. "order 17 was placed". Has a **value** (the data), optionally a **key**, a **timestamp**, and **headers**. | Today we only use the value. |
| **Topic** | A named stream of messages, like a folder or table name: `orders`, `payments`. | Producers write *to a topic*; consumers read *from a topic*. |
| **Partition** | A topic is split into N separate lists. Each one is an **append-only log**: new messages go on the end; nothing is edited or removed in the middle. | Partitions exist so work can be spread across servers and consumers. **Order is guaranteed only inside one partition**, never across partitions. |
| **Offset** | The position number of a message *inside one partition*: 0, 1, 2, ... Never reused, never changed. | Offsets are **per partition**: partition 0 and partition 1 each have their own offset 0. |
| **Segment** | A partition is stored on disk as a series of files; each file is a *segment*. When one gets full, Kafka starts the next. | This is how Kafka deletes old data cheaply: it removes whole files. |

### The address of any message
```text
   topic  +  partition  +  offset      e.g.   orders / 0 / 2   =  "order-D"
```
Reading a message does **not** remove it. The message stays until the retention rule (Step 10) deletes it. This is the big difference from a queue.

## Step 2 — Create a topic

Topics can be created by a program. We ask for **3 partitions** and **replication factor 1**.

- `num_partitions=3` — split the topic into 3 logs.
- `replication_factor=1` — keep 1 copy of each partition. (Production uses 3. We have one server, so 1 is the maximum.)

`create_topics` returns immediately with a *future* per topic (a placeholder for "the answer will arrive later"). Calling `.result()` waits for the answer and raises an error if creation failed.

We first delete any old topic of the same name so you can re-run this notebook safely.

In [4]:
def recreate_topic(name, partitions, config=None):
    """Delete the topic if it exists, wait until it is gone, then create it fresh."""
    if name in admin.list_topics(timeout=10).topics:
        admin.delete_topics([name])[name].result()
        while name in admin.list_topics(timeout=10).topics:
            time.sleep(0.5)
    new_topic = NewTopic(name, num_partitions=partitions, replication_factor=1, config=config or {})
    admin.create_topics([new_topic])[name].result()
    print(f"Created topic '{name}' with {partitions} partition(s)")


TOPIC = "orders"
recreate_topic(TOPIC, partitions=3)

Created topic 'orders' with 3 partition(s)


## Step 3 — Inspect the topic

`kafka-topics.sh --describe` is Kafka's own tool for this.

In [7]:
kafka_tool("kafka-topics.sh", "--bootstrap-server", "localhost:9092", "--describe", "--topic", TOPIC)

Topic: orders	TopicId: Pdeb4FBNTOq_dXszBlAsqw	PartitionCount: 3	ReplicationFactor: 1	Configs: min.insync.replicas=1
	Topic: orders	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: orders	Partition: 1	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: orders	Partition: 2	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-topics.sh', '--bootstrap-server', 'localhost:9092', '--describe', '--topic', 'orders'], returncode=0, stdout='Topic: orders\tTopicId: Pdeb4FBNTOq_dXszBlAsqw\tPartitionCount: 3\tReplicationFactor: 1\tConfigs: min.insync.replicas=1\n\tTopic: orders\tPartition: 0\tLeader: 1\tReplicas: 1\tIsr: 1\tElr: \tLastKnownElr: \n\tTopic: orders\tPartition: 1\tLeader: 1\tReplicas: 1\tIsr: 1\tElr: \tLastKnownElr: \n\tTopic: orders\tPartition: 2\tLeader: 1\tReplicas: 1\tIsr: 1\tElr: \tLastKnownElr: \n', stderr='')

**Reading the output**

| Field | Meaning |
|---|---|
| `TopicId` | Unique ID. If you delete and recreate a topic, the ID changes even though the name is the same. |
| `PartitionCount: 3` | Three logs, numbered 0, 1, 2. |
| `ReplicationFactor: 1` | One copy of each. |
| `Leader: 1` | Broker (server) number 1 handles all reads and writes for this partition. With many brokers, different partitions have different leaders; that spreads the load. |
| `Replicas: 1` | Brokers holding a copy. |
| `Isr: 1` | **In-Sync Replicas**: copies that are fully caught up with the leader. In production you want `Isr` to equal `Replicas`. If it is shorter, a copy is lagging or its server is down. |
| `Elr`, `LastKnownElr` | Newer Kafka bookkeeping for "eligible" leader replicas. Ignore for now. |

## Step 4 — Put 6 messages in

We are previewing the producer here; **Notebook 03 explains it properly**. For now: a `Producer` sends messages; `flush()` waits until Kafka confirms them; the callback prints the confirmation.

Normally Kafka *chooses* the partition. To make today's picture predictable, **we choose it** with `partition=`.

In [8]:
producer = Producer({"bootstrap.servers": BOOTSTRAP})

# Callback function. Once the Kafka server successfully saves the message (or fails trying), it triggers this report function so your app knows what happened.
def report(error, message):
    """Called once per message when Kafka confirms (or rejects) it."""
    if error:
        print("FAILED:", error)
    else:
        print(f"stored: {message.topic()} / partition {message.partition()} / offset {message.offset()}  value={message.value().decode()}")


what_to_send = [(0, "order-A"), (0, "order-B"), (1, "order-C"), (0, "order-D"), (2, "order-E"), (1, "order-F")]

for partition, value in what_to_send:
    # TOPIC (The Destination): The name of the topic (the "highway") where you are sending the data, such as "orders".
    # value (The Payload): The actual message you are sending. This is usually your raw data, like a JSON string containing the event details.
    # partition (The Specific Lane): This forces the message into an exact partition number (e.g., 0, 1, or 2). By providing this, you are bypassing Kafka's automatic routing and manually dictating exactly which "toll lane" the message must use.
    producer.produce(TOPIC, value=value, partition=partition, on_delivery=report)

producer.flush()

stored: orders / partition 1 / offset 0  value=order-C
stored: orders / partition 1 / offset 1  value=order-F
stored: orders / partition 2 / offset 0  value=order-E
stored: orders / partition 0 / offset 0  value=order-A
stored: orders / partition 0 / offset 1  value=order-B
stored: orders / partition 0 / offset 2  value=order-D


0

**Look at the offsets.** Messages A, B, D went to partition 0 and got offsets 0, 1, 2. Message C went to partition 1 and got offset **0**; its own list started at zero.

(The print order may differ from the send order: the callbacks are grouped by partition. That is itself a lesson: nothing orders messages *across* partitions.)

```text
 partition 0:  [A:0][B:1][D:2]
 partition 1:  [C:0][F:1]
 partition 2:  [E:0]
```

## Step 5 — Ask Kafka: how far has each partition grown?

`kafka-get-offsets.sh` prints the **end offset**: the offset the *next* message will receive. Output format: `topic:partition:offset`.

In [9]:
print("Next offset to be written (the 'end'):")
kafka_tool("kafka-get-offsets.sh", "--bootstrap-server", "localhost:9092", "--topic", TOPIC, "--time", "latest")
print()
print("First offset still available (the 'start'):")
kafka_tool("kafka-get-offsets.sh", "--bootstrap-server", "localhost:9092", "--topic", TOPIC, "--time", "earliest")

Next offset to be written (the 'end'):
orders:0:3
orders:1:2
orders:2:1

First offset still available (the 'start'):
orders:0:0
orders:1:0
orders:2:0


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-get-offsets.sh', '--bootstrap-server', 'localhost:9092', '--topic', 'orders', '--time', 'earliest'], returncode=0, stdout='orders:0:0\norders:1:0\norders:2:0\n', stderr='')

Expected: ends `0:3`, `1:2`, `2:1` and starts all `0`. Number of messages in a partition = end - start. Once old data is deleted (Step 10), the *start* moves up while offsets are never renumbered.

## Step 6 — Read a message by its address

The console consumer is Kafka's built-in reader. We give it the address: partition 0, starting at offset 1, and stop after 2 messages.

In [10]:
kafka_tool(
    "kafka-console-consumer.sh",
    "--bootstrap-server", "localhost:9092",
    "--topic", TOPIC,
    "--partition", "0",
    "--offset", "1",
    "--max-messages", "2",
    "--formatter-property", "print.offset=true",
)

Offset:1	order-B
Offset:2	order-D
Processed a total of 2 messages


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-console-consumer.sh', '--bootstrap-server', 'localhost:9092', '--topic', 'orders', '--partition', '0', '--offset', '1', '--max-messages', '2', '--formatter-property', 'print.offset=true'], returncode=0, stdout='Offset:1\torder-B\nOffset:2\torder-D\n', stderr='Processed a total of 2 messages\n')

Expected: `Offset:1  order-B` and `Offset:2  order-D`. Run the cell again: you get the same answer. **Reading does not consume.** (How consumers remember where they were is Notebook 04.)

## Step 7 — Open the files on disk

Everything above is stored as ordinary files. Each partition is a **folder** named `<topic>-<partition>`.

In [11]:
print("Partition folders:")
in_container("sh", "-c", f"ls {DATA_DIR} | grep {TOPIC}")
print()
print(f"Inside {TOPIC}-0:")
in_container("ls", "-l", f"{DATA_DIR}/{TOPIC}-0")

Partition folders:
orders-0
orders-1
orders-2

Inside orders-0:
total 12
-rw-r--r-- 1 appuser appuser 10485760 Oct 15 07:12 00000000000000000000.index
-rw-r--r-- 1 appuser appuser      103 Oct 15 07:52 00000000000000000000.log
-rw-r--r-- 1 appuser appuser 10485756 Oct 15 07:12 00000000000000000000.timeindex
-rw-r--r-- 1 appuser appuser        8 Oct 15 07:12 leader-epoch-checkpoint
-rw-r--r-- 1 appuser appuser       43 Oct 15 07:12 partition.metadata


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', 'ls', '-l', '/var/lib/kafka/data/orders-0'], returncode=0, stdout='total 12\n-rw-r--r-- 1 appuser appuser 10485760 Oct 15 07:12 00000000000000000000.index\n-rw-r--r-- 1 appuser appuser      103 Oct 15 07:52 00000000000000000000.log\n-rw-r--r-- 1 appuser appuser 10485756 Oct 15 07:12 00000000000000000000.timeindex\n-rw-r--r-- 1 appuser appuser        8 Oct 15 07:12 leader-epoch-checkpoint\n-rw-r--r-- 1 appuser appuser       43 Oct 15 07:12 partition.metadata\n', stderr='')

| File | What it holds |
|---|---|
| `00000000000000000000.log` | **The messages themselves.** The name is the offset of the first message in this file (here 0). |
| `00000000000000000000.index` | A small lookup: *offset -> byte position* in the `.log`. Lets Kafka jump to offset 1,000,000 without reading everything before it. |
| `00000000000000000000.timeindex` | Lookup: *timestamp -> offset*. Powers "start reading from yesterday 14:00". |
| `leader-epoch-checkpoint` | Records which broker was leader when. Used to keep copies consistent after failures. |
| `partition.metadata` | The topic's unique ID, so the folder can't be confused with a deleted-and-recreated topic. |

The `.log` file is tiny here (a few hundred bytes), but the `.index` files may look much larger: Kafka reserves space for them in advance.

## Step 8 — Decode the `.log` file

Kafka ships a tool that reads a `.log` file and prints what is inside. This is the closest you can get to the truth.

In [12]:
kafka_tool(
    "kafka-dump-log.sh",
    "--files", f"{DATA_DIR}/{TOPIC}-0/00000000000000000000.log",
    "--print-data-log",
)

Dumping /var/lib/kafka/data/orders-0/00000000000000000000.log
Log starting offset: 0
baseOffset: 0 lastOffset: 2 count: 3 baseSequence: -1 lastSequence: -1 producerId: -1 producerEpoch: -1 partitionLeaderEpoch: 0 isTransactional: false isControl: false deleteHorizonMs: OptionalLong.empty position: 0 CreateTime: 1792050733382 size: 103 magic: 2 compresscodec: none crc: 2548448774 isvalid: true
| offset: 0 CreateTime: 1792050733382 keySize: -1 valueSize: 7 sequence: -1 headerKeys: [] payload: order-A
| offset: 1 CreateTime: 1792050733382 keySize: -1 valueSize: 7 sequence: -1 headerKeys: [] payload: order-B
| offset: 2 CreateTime: 1792050733382 keySize: -1 valueSize: 7 sequence: -1 headerKeys: [] payload: order-D


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-dump-log.sh', '--files', '/var/lib/kafka/data/orders-0/00000000000000000000.log', '--print-data-log'], returncode=0, stdout='Dumping /var/lib/kafka/data/orders-0/00000000000000000000.log\nLog starting offset: 0\nbaseOffset: 0 lastOffset: 2 count: 3 baseSequence: -1 lastSequence: -1 producerId: -1 producerEpoch: -1 partitionLeaderEpoch: 0 isTransactional: false isControl: false deleteHorizonMs: OptionalLong.empty position: 0 CreateTime: 1792050733382 size: 103 magic: 2 compresscodec: none crc: 2548448774 isvalid: true\n| offset: 0 CreateTime: 1792050733382 keySize: -1 valueSize: 7 sequence: -1 headerKeys: [] payload: order-A\n| offset: 1 CreateTime: 1792050733382 keySize: -1 valueSize: 7 sequence: -1 headerKeys: [] payload: order-B\n| offset: 2 CreateTime: 1792050733382 keySize: -1 valueSize: 7 sequence: -1 headerKeys: [] payload: order-D\n', stderr='')

**Reading the output**

| You see | Meaning |
|---|---|
| `baseOffset: 0` | Messages are stored in **batches**. This batch starts at offset 0. |
| `lastOffset`, `count` | Last offset in the batch and how many messages it holds. (A producer that sends messages quickly puts several in one batch.) |
| `position: 0` | Byte position of the batch in the file. This is what the `.index` file stores. |
| `CreateTime` | Timestamp, in milliseconds since 1970. |
| `size` | Bytes taken by the batch. |
| `keySize: -1` | The message has no key (`-1` means "none"). Keys arrive in Notebook 03. |
| `| offset: 0 ... payload: order-A` | The individual messages, with their offsets and contents. |

You are now looking at exactly what Kafka keeps on disk: nothing but a sequence of messages with increasing offsets.

## Step 9 — Watch a log split into segments

A segment file cannot grow forever. The setting `segment.bytes` is the maximum size of one. Its default is 1 GB, so we would never see a split in a demo. Kafka's smallest allowed value is **1 MiB** (1,048,576 bytes).

We create a second topic, `tiny`, with 1 MiB segments, and write 30 messages of about 100 KB each (about 3 MB in total). That should fill roughly three segments.

In [ ]:
recreate_topic("tiny", partitions=1, config={"segment.bytes": str(1024 * 1024)})

for number in range(30):
    producer.produce("tiny", value=f"message {number:02d} ".encode() + b"x" * 100_000)
    producer.poll(0)          # let the producer process confirmations while we keep sending
producer.flush()

in_container("sh", "-c", f"ls -l {DATA_DIR}/tiny-0 | grep '.log'")

**Look at the `.log` file names.** Each name is the offset of the *first message inside that file*. A name like `00000000000000000009.log` means "this segment starts at offset 9; everything before it lives in the earlier file".

Expected (real output from a test run):
```text
 tiny-0/
   00000000000000000000.log     ~900 KB    offsets  0 ..  8
   00000000000000000009.log     ~900 KB    offsets  9 .. 17
   00000000000000000018.log     ~900 KB    offsets 18 .. 26
   00000000000000000027.log     ~300 KB    offsets 27 ..      <- the ACTIVE segment, still being written
```
Why 9 messages per file and not 10? Each message is about 100 KB. A 10th would push the file past the 1 MiB limit, so Kafka starts a new file instead.

Only the last file (the **active segment**) receives new messages. Older ones are never modified, which makes them safe to delete or copy. (Your exact split points may differ slightly.)

> A small `segment.bytes` is for learning only. In production, tiny segments mean thousands of open files and a slow broker.

## Step 10 — Retention: how Kafka deletes

Kafka does **not** delete a message when it is read. It deletes by **age or size**, and only **whole segments**, never single messages. Let's look at the retention settings of our topic.

In [13]:
interesting = ["retention.ms", "retention.bytes", "segment.bytes", "segment.ms", "cleanup.policy", "min.insync.replicas", "max.message.bytes"]

result = kafka_tool(
    "kafka-configs.sh", "--bootstrap-server", "localhost:9092",
    "--entity-type", "topics", "--entity-name", TOPIC, "--describe", "--all",
    show=False,
)
for line in result.stdout.splitlines():
    name = line.strip().split("=")[0]
    if name in interesting:
        print(line.strip().split(" sensitive=")[0])

cleanup.policy=delete
max.message.bytes=1048588
min.insync.replicas=1
retention.bytes=-1
retention.ms=604800000
segment.bytes=1073741824
segment.ms=604800000


| Setting | Default | Meaning |
|---|---|---|
| `retention.ms` | `604800000` | Keep data for **7 days** (604,800,000 ms). Older segments are deleted. `-1` = keep forever. |
| `retention.bytes` | `-1` | Optional size cap per **partition**. `-1` = no cap. If both are set, whichever limit is hit first deletes data. |
| `segment.bytes` | `1073741824` | Max size of one segment file (1 GB). |
| `segment.ms` | `604800000` | Even if not full, start a new segment after 7 days. |
| `cleanup.policy` | `delete` | `delete` = remove old segments by the rules above. The alternative, `compact`, keeps only the *latest message per key*. Used for "current state" topics. |
| `min.insync.replicas` | `1` | Covered in Notebook 05. |
| `max.message.bytes` | about 1 MB | Largest single message Kafka accepts. |

**Why whole segments only?** Deleting a file is instant. Editing the middle of a file would be slow and would break offsets. This is why the *active* segment can never be deleted, and why data may live a little longer than `retention.ms` says: a segment is deleted only when its **newest** message is old enough.

To change a setting on an existing topic later: `kafka-configs.sh --alter --add-config retention.ms=3600000`.

## Step 11 — Clean up

We keep `orders` (Notebook 03 will use it), and remove the experiment topic.

In [14]:
admin.delete_topics(["tiny"])["tiny"].result()
print("Deleted 'tiny'. Topics now:", sorted(t for t in admin.list_topics(timeout=10).topics if not t.startswith("__")))

KafkaException: KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Broker: Unknown topic or partition"}

## Checkpoint — answer without scrolling up

1. Name the five words and give a one-line meaning of each.
2. How do you write the address of a message?
3. Partition 0 has 3 messages and partition 1 has 2. What offset does the next message to partition 1 get?
4. Can you tell, from offsets alone, which of two messages in *different* partitions came first? Why?
5. What does the number in `00000000000000000012.log` mean?
6. A consumer reads message 5. Is the message still there afterwards? What eventually removes it?
7. Why does Kafka delete whole segments rather than single messages?

## Glossary so far

| Term | One line |
|---|---|
| Broker | One Kafka server |
| Topic | Named stream of messages |
| Partition | One append-only log inside a topic |
| Offset | Position of a message inside one partition |
| Segment | One file piece of a partition's log |
| Active segment | The newest segment, the only one being written |
| Leader | The broker that handles reads/writes for a partition |
| ISR | Replicas that are caught up with the leader |
| Retention | Rule deciding when old segments are deleted |

**Next: Notebook 03 — the Producer.** How a message travels from Python to a partition, how keys choose partitions, and how you know it arrived.